# Day 3 · Notebook 2 — Student lab (client side)

Server-side exercises (adding endpoints, auth, logging) are in **`LAB.md`**.
These exercises only need a running API on `http://localhost:5000`.

Each exercise: try it first, then open the **✅ Solution** cell.

In [ ]:
import subprocess, sys, time, io
from pathlib import Path
import numpy as np, pandas as pd, requests
from PIL import Image

API = "http://localhost:5000"
def api_up():
    try: return requests.get(f"{API}/health", timeout=1).ok
    except requests.ConnectionError: return False

server = None
if not api_up():
    log_file = open("api_server.log", "w")
    server = subprocess.Popen([sys.executable, "app.py"], cwd=Path("app/api").resolve(),
                              stdout=log_file, stderr=subprocess.STDOUT)
    for _ in range(60):
        if api_up() or server.poll() is not None: break
        time.sleep(0.5)
    if not api_up():
        log_file.close()
        raise RuntimeError("API failed to start:\n" + Path("api_server.log").read_text()
                           + f"\nKernel Python: {sys.executable} - is this the course venv?")
samples = sorted(Path("app/ui/samples").glob("*.png"))
def to_bytes(im):
    b = io.BytesIO(); im.save(b, format="PNG"); return b.getvalue()
print("API up:", api_up(), "|", len(samples), "samples")

## Exercise 1 (10 min) — Write a tiny Python client class

Write a class `CifarClient` with methods `health()`, `predict(path, model="int8")` and `compare(path)`.
It should raise a clear exception when the API returns a non-200 status.

In [ ]:
# TODO: your code here

### ✅ Solution (reveal after trying)

In [ ]:
class CifarClient:
    def __init__(self, base_url="http://localhost:5000", timeout=10):
        self.base, self.timeout = base_url.rstrip("/"), timeout
        self.s = requests.Session()             # re-uses TCP connections -> lower latency

    def _check(self, r):
        if r.status_code != 200:
            raise RuntimeError(f"API error {r.status_code}: {r.json().get('error')}")
        return r.json()

    def health(self):
        return self._check(self.s.get(f"{self.base}/health", timeout=self.timeout))

    def predict(self, path, model="int8", top_k=3):
        with open(path, "rb") as f:
            return self._check(self.s.post(f"{self.base}/predict", params={"model": model, "top_k": top_k},
                                           files={"file": f}, timeout=self.timeout))

    def compare(self, path):
        with open(path, "rb") as f:
            return self._check(self.s.post(f"{self.base}/compare", files={"file": f}, timeout=self.timeout))

client = CifarClient()
print(client.health(), client.predict(samples[0])["prediction"])
try:
    client.predict(samples[0], model="nope")
except RuntimeError as e:
    print("caught:", e)

## Exercise 2 (15 min) — How robust is the model? Rotation & blur

For every sample image, apply (a) rotation by 0/15/30/45/90° and (b) Gaussian blur radius 0/1/2/3,
send to the API and compute accuracy per perturbation level for **both** fp32 and int8.
Does quantization make the model *less robust*?

Hint: `im.rotate(angle)`, `from PIL import ImageFilter; im.filter(ImageFilter.GaussianBlur(r))`

In [ ]:
# TODO: your code here

### ✅ Solution (reveal after trying)

In [ ]:
from PIL import ImageFilter
rows = []
for kind, levels, fn in [("rotate", [0, 15, 30, 45, 90], lambda im, a: im.rotate(a)),
                         ("blur", [0, 1, 2, 3], lambda im, r: im.filter(ImageFilter.GaussianBlur(r)))]:
    for lvl in levels:
        for model in ["fp32", "int8"]:
            correct = 0
            for p in samples:
                im = fn(Image.open(p).convert("RGB"), lvl)
                pred = requests.post(f"{API}/predict", params={"model": model},
                                     files={"file": to_bytes(im)}).json()["prediction"]
                correct += pred == p.stem.split("_")[0]
            rows.append({"perturbation": kind, "level": lvl, "model": model, "acc": correct / len(samples)})
rob = pd.DataFrame(rows)
rob.pivot_table(index=["perturbation", "level"], columns="model", values="acc")

## Exercise 3 (10 min) — Confidence threshold ("I'm not sure")

Write `safe_predict(path, threshold)` that returns `"unsure"` when the top-1 confidence is below `threshold`.
Using the sample images **plus** 10 random-noise images, pick a threshold that rejects most noise
while keeping most real images. Print a small table threshold → (% real accepted, % noise rejected).

In [ ]:
# TODO: your code here

### ✅ Solution (reveal after trying)

In [ ]:
rng = np.random.default_rng(0)
noise_imgs = [Image.fromarray((rng.random((32, 32, 3)) * 255).astype("uint8")) for _ in range(10)]
real_conf = [requests.post(f"{API}/predict", files={"file": p.read_bytes()}).json()["confidence"] for p in samples]
noise_conf = [requests.post(f"{API}/predict", files={"file": to_bytes(im)}).json()["confidence"] for im in noise_imgs]

table = []
for t in [0.3, 0.5, 0.6, 0.7, 0.8, 0.9]:
    table.append({"threshold": t,
                  "real_accepted_%": 100 * np.mean(np.array(real_conf) >= t),
                  "noise_rejected_%": 100 * np.mean(np.array(noise_conf) < t)})
print(pd.DataFrame(table))
print("\nNote: softmax confidence is a weak OOD signal - discuss why (Day 5 question bank).")

## Exercise 4 (10 min) — Session re-use & connection overhead

Measure mean latency of 50 requests using (a) `requests.post` each time (new TCP connection)
vs (b) a single `requests.Session()`. Explain the difference.

In [ ]:
# TODO: your code here

### ✅ Solution (reveal after trying)

In [ ]:
payload = samples[0].read_bytes()
def timed(fn, n=50):
    t0 = time.perf_counter()
    for _ in range(n): fn()
    return (time.perf_counter() - t0) * 1000 / n

no_session = timed(lambda: requests.post(f"{API}/predict", files={"file": payload}))
s = requests.Session()
with_session = timed(lambda: s.post(f"{API}/predict", files={"file": payload}))
print(f"new connection each time: {no_session:.1f} ms | keep-alive session: {with_session:.1f} ms")
print("The session keeps the TCP connection open (HTTP keep-alive): no handshake per request.")

In [ ]:
if server is not None:
    server.terminate(); server.wait(timeout=10); print("API stopped")